# LeetCode 1737 — Change Minimum Characters to Satisfy One of Three Conditions bằng Python

Link đề: https://leetcode.com/problems/change-minimum-characters-to-satisfy-one-of-three-conditions/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Counting, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def minCharacters(self, a: str, b: str) -> int:
        ...
```

Lộ trình: **vét cạn mọi chuỗi đích** trên bảng chữ cái thu nhỏ (kiểm chứng) → **thử mọi ngưỡng** và quét lại chuỗi (`O(26·(|a|+|b|))`) → **đếm tần suất 26 chữ cái** rồi tính từ bảng đếm (`O(|a|+|b| + 26²)`) → **tổng tiền tố trên bảng đếm** (`O(|a|+|b| + 26)`).

## 1. Mục tiêu học

1. Nhận ra điều kiện "mọi chữ của `a` < mọi chữ của `b`" tương đương với **tồn tại ngưỡng** `t` sao cho mọi chữ của `a` `≤ t` và mọi chữ của `b` `> t`.
2. Biến bài toán tối ưu trên chuỗi thành tính toán trên **bảng tần suất 26 phần tử**.
3. Dùng **tổng tiền tố** trên bảng tần suất để đếm "số ký tự lớn hơn / nhỏ hơn hoặc bằng `t`" trong `O(1)`.
4. Xử lý biên: ngưỡng chỉ chạy từ `'a'` tới `'y'` (không có chữ nào lớn hơn `'z'`).
5. Kiểm chứng bằng **vét cạn trên bảng chữ cái thu nhỏ** — kỹ thuật tổng quát hóa tham số để có thể brute force.

## 2. Đề bài & ví dụ

Cho hai chuỗi `a` và `b` gồm chữ cái thường. Một thao tác: đổi **một ký tự bất kỳ** trong `a` hoặc `b` thành **một chữ cái thường bất kỳ**.

Mục tiêu: thỏa **một trong ba** điều kiện:
1. Mọi chữ trong `a` **nhỏ hơn nghiêm ngặt** mọi chữ trong `b` (theo bảng chữ cái).
2. Mọi chữ trong `b` nhỏ hơn nghiêm ngặt mọi chữ trong `a`.
3. Cả `a` và `b` chỉ gồm **một** chữ cái duy nhất (cùng một chữ cho cả hai chuỗi).

Trả về số thao tác **ít nhất**.

**Ví dụ 1:** `a = "aba"`, `b = "caa"` → `2`.
- Điều kiện 1: đổi `b` thành `"ccc"` — 2 thao tác.
- Điều kiện 2: đổi `a` thành `"bbb"`, `b` thành `"aaa"` — 3 thao tác.
- Điều kiện 3: đổi `a` thành `"aaa"`, `b` thành `"aaa"` — 2 thao tác.

**Ví dụ 2:** `a = "dabadd"`, `b = "cda"` → `3` (điều kiện 1: đổi `b` thành `"eee"`).

**Ràng buộc:**
- `1 ≤ a.length, b.length ≤ 10⁵`
- `a`, `b` chỉ gồm chữ cái thường.

In [1]:
import random
import time
from itertools import product
from collections import Counter
from typing import List

examples = [
    (("aba", "caa"), 2),
    (("dabadd", "cda"), 3),
]
for (a, b), want in examples:
    print(f"a={a!r}, b={b!r} -> kỳ vọng {want}")

a='aba', b='caa' -> kỳ vọng 2
a='dabadd', b='cda' -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1 So sánh ký tự và `ord`

`'a' < 'b' < … < 'z'`. `ord(ch) − 97` cho chỉ số `0..25`.

In [2]:
print('c' > 'a', ord('c') - 97, chr(97 + 4))

True 2 e


### 3.2 Bảng tần suất 26 phần tử

In [3]:
def freq(s, K=26):
    f = [0] * K
    for ch in s:
        f[ord(ch) - 97] += 1
    return f

print(freq("dabadd")[:6], "(a, b, c, d, e, f)")

[2, 1, 0, 3, 0, 0] (a, b, c, d, e, f)


### 3.3 Tổng tiền tố

`P[i] = f[0] + … + f[i−1]` ⇒ số ký tự `≤ t` là `P[t + 1]`, số ký tự `> t` là `len − P[t + 1]`.

In [4]:
from itertools import accumulate
f = freq("dabadd")
P = [0] + list(accumulate(f))
t = 1    # 'b'
print(f"số ký tự <= 'b': {P[t + 1]}, số ký tự > 'b': {len('dabadd') - P[t + 1]}")

số ký tự <= 'b': 3, số ký tự > 'b': 3


### 3.4 "max(a) < min(b)" ⇔ "tồn tại ngưỡng t"

Mọi chữ của `a` nhỏ hơn mọi chữ của `b` ⇔ `max(a) < min(b)` ⇔ tồn tại `t` với `max(a) ≤ t < min(b)` (chọn `t = max(a)`). Vì `min(b) ≤ 'z'`, ngưỡng `t ≤ 'y'`.

In [5]:
def cond1(a, b):
    return max(a) < min(b)
def cond3(a, b):
    return len(set(a) | set(b)) == 1
print(cond1("aab", "ccd"), cond1("aac", "ccd"), cond3("aaa", "aa"), cond3("aaa", "ab"))

True False True False


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 2 `a = "dabadd"` (tần suất `a:2, b:1, d:3`), `b = "cda"` (`a:1, c:1, d:1`).

**Điều kiện 1 với ngưỡng `t`:** phải đổi mọi ký tự của `a` lớn hơn `t` và mọi ký tự của `b` nhỏ hơn hoặc bằng `t`.

| t | a có bao nhiêu > t | b có bao nhiêu ≤ t | tổng |
|---|---|---|---|
| a | 4 (b, d, d, d) | 1 (a) | 5 |
| b | 3 (d, d, d) | 1 | 4 |
| c | 3 | 2 (a, c) | 5 |
| d | 0 | 3 | **3** |
| e…y | 0 | 3 | 3 |

**Điều kiện 2** là đối xứng (đổi vai `a`, `b`). **Điều kiện 3** với chữ `c`: đổi mọi ký tự khác `c` ⇒ `|a| + |b| − (fa[c] + fb[c])`; tốt nhất với `c = 'd'`: `9 − 4 = 5`.

Đáp án `min(3, …, 5) = 3`.

## 5. Cách 1 — Brute Force / baseline

### 5.1 Vét cạn mọi chuỗi đích (bảng chữ cái thu nhỏ `K` chữ)

Để kiểm chứng, ta **tổng quát hóa** bài toán với bảng chữ cái `K` chữ đầu tiên (đề gốc `K = 26`). Với `K` nhỏ và chuỗi ngắn, thử **mọi** cặp chuỗi đích `(a', b')`, tính số vị trí khác nhau và kiểm tra một trong ba điều kiện.

In [6]:
def min_chars_exhaustive(a, b, K):
    letters = [chr(97 + i) for i in range(K)]
    best = None
    for a2 in product(letters, repeat=len(a)):
        ca = sum(x != y for x, y in zip(a, a2))
        if best is not None and ca >= best:
            continue
        for b2 in product(letters, repeat=len(b)):
            if cond1(a2, b2) or cond1(b2, a2) or len(set(a2) | set(b2)) == 1:
                cost = ca + sum(x != y for x, y in zip(b, b2))
                if best is None or cost < best:
                    best = cost
    return best

print(min_chars_exhaustive("aba", "caa", 4), "(K = 4, ví dụ 1 vẫn cho 2)")

2 (K = 4, ví dụ 1 vẫn cho 2)


**Độ phức tạp:** `K^{|a|+|b|}` cặp chuỗi — chỉ để kiểm chứng (`K ≤ 4`, độ dài ≤ 3).

### 5.2 Cách 1 (baseline thật) — thử mọi ngưỡng, quét lại chuỗi

Với mỗi ngưỡng `t ∈ [0, K−2]` đếm trực tiếp trên chuỗi; với mỗi chữ `c` đếm trực tiếp cho điều kiện 3.

In [7]:
def min_chars_scan(a, b, K=26):
    best = len(a) + len(b)
    for t in range(K - 1):
        T = chr(97 + t)
        c1 = sum(ch > T for ch in a) + sum(ch <= T for ch in b)     # a <= t < b
        c2 = sum(ch > T for ch in b) + sum(ch <= T for ch in a)     # b <= t < a
        best = min(best, c1, c2)
    for c in range(K):
        C = chr(97 + c)
        best = min(best, len(a) + len(b) - a.count(C) - b.count(C))
    return best

for (a, b), want in examples:
    print(min_chars_scan(a, b), "kỳ vọng", want)

2 kỳ vọng 2
3 kỳ vọng 3


**Độ phức tạp:** Time `O(K · (|a| + |b|))` ≈ `26 · 2·10⁵ = 5.2·10⁶` phép — chạy được nhưng chậm trong Python (generator). Space `O(1)`.

**Nút thắt:** mỗi ngưỡng quét lại toàn bộ chuỗi, trong khi thông tin cần chỉ là **số ký tự mỗi loại**.

In [8]:
random.seed(0)
A = "".join(random.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(10**5))
B = "".join(random.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(10**5))
t0 = time.perf_counter(); r = min_chars_scan(A, B)
print(f"|a| = |b| = 1e5: {r}, {time.perf_counter() - t0:.3f}s")

|a| = |b| = 1e5: 99836, 0.258s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** chi phí mọi điều kiện chỉ phụ thuộc **bảng tần suất** `fa`, `fb`:
- **Điều kiện 1** (ngưỡng `t`, `a ≤ t < b`): `cost1(t) = Σ_{c > t} fa[c] + Σ_{c ≤ t} fb[c]`.
- **Điều kiện 2** (ngưỡng `t`, `b ≤ t < a`): `cost2(t) = Σ_{c > t} fb[c] + Σ_{c ≤ t} fa[c]`.
- **Điều kiện 3** (chữ `c`): `cost3(c) = |a| + |b| − fa[c] − fb[c]`.

Với tổng tiền tố `PA`, `PB`: `cost1(t) = (|a| − PA[t+1]) + PB[t+1]`, `cost2(t) = (|b| − PB[t+1]) + PA[t+1]`.

**Đáp án:** `min( min_{t=0..24} cost1(t), min_{t=0..24} cost2(t), min_{c=0..25} cost3(c) )`.

**Vì sao đổi ký tự "vi phạm" là đủ và tối ưu?** Với ngưỡng `t` cố định, mỗi ký tự vi phạm **bắt buộc** phải đổi, và đổi nó thành một chữ hợp lệ bất kỳ (ví dụ `t` cho `a`, `t+1` cho `b`) là đủ; ký tự không vi phạm thì không cần đổi.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Bảng tần suất, cộng trực tiếp cho mỗi ngưỡng (`O(n + K²)`)

In [9]:
def min_chars_freq(a, b, K=26):
    fa, fb = freq(a, K), freq(b, K)
    best = len(a) + len(b)
    for t in range(K - 1):
        a_gt = sum(fa[t + 1:]); a_le = sum(fa[:t + 1])
        b_gt = sum(fb[t + 1:]); b_le = sum(fb[:t + 1])
        best = min(best, a_gt + b_le, b_gt + a_le)
    for c in range(K):
        best = min(best, len(a) + len(b) - fa[c] - fb[c])
    return best

for (a, b), want in examples:
    print(min_chars_freq(a, b), "kỳ vọng", want)

2 kỳ vọng 2
3 kỳ vọng 3


**Độ phức tạp:** Time `O(|a| + |b| + K²)`, Space `O(K)`.

### 7.2 Cách 3 — Tổng tiền tố trên bảng tần suất (`O(n + K)`)

Từng đoạn:
1. `fa`, `fb` — tần suất.
2. Duyệt `t = 0..K−2`, cộng dồn `pa += fa[t]`, `pb += fb[t]` (số ký tự `≤ t`).
3. `cost1 = (|a| − pa) + pb`, `cost2 = (|b| − pb) + pa`.
4. Điều kiện 3 như trên.

In [10]:
def min_chars_prefix(a, b, K=26):
    fa, fb = freq(a, K), freq(b, K)
    na, nb = len(a), len(b)
    best = na + nb
    pa = pb = 0
    for t in range(K - 1):
        pa += fa[t]
        pb += fb[t]
        best = min(best, (na - pa) + pb, (nb - pb) + pa)
    for c in range(K):
        best = min(best, na + nb - fa[c] - fb[c])
    return best

for (a, b), want in examples:
    print(min_chars_prefix(a, b), "kỳ vọng", want)

2 kỳ vọng 2
3 kỳ vọng 3


**Độ phức tạp:** Time `O(|a| + |b| + K)`, Space `O(K)`.

### 7.3 Cách 4 — Đếm bằng `Counter` / `str.count` (nhanh trong Python)

`str.count` chạy bằng C; 26 lần `count` trên mỗi chuỗi nhanh hơn một vòng `for` Python qua `10⁵` ký tự.

In [11]:
def min_chars_count(a, b):
    letters = "abcdefghijklmnopqrstuvwxyz"
    fa = [a.count(c) for c in letters]
    fb = [b.count(c) for c in letters]
    na, nb = len(a), len(b)
    best = min(na + nb - x - y for x, y in zip(fa, fb))
    pa = pb = 0
    for t in range(25):
        pa += fa[t]; pb += fb[t]
        best = min(best, na - pa + pb, nb - pb + pa)
    return best

for (a, b), want in examples:
    print(min_chars_count(a, b), "kỳ vọng", want)

2 kỳ vọng 2
3 kỳ vọng 3


**Độ phức tạp:** Time `O(26 · (|a| + |b|))` nhưng hằng số rất nhỏ (C), Space `O(26)`.

## 8. TRACE từng bước

Trace Cách 3 trên ví dụ 1 `a = "aba"`, `b = "caa"` (chỉ in các ngưỡng đầu và chữ có mặt):

In [12]:
def trace_prefix(a, b, show_t=5):
    fa, fb = freq(a), freq(b); na, nb = len(a), len(b)
    print("fa (a..e):", fa[:5], "| fb (a..e):", fb[:5])
    best = na + nb; pa = pb = 0
    for t in range(25):
        pa += fa[t]; pb += fb[t]
        c1 = na - pa + pb; c2 = nb - pb + pa
        best = min(best, c1, c2)
        if t < show_t:
            print(f"t={chr(97 + t)}: a<=t: {pa}, b<=t: {pb} | ĐK1 (a<=t<b) = {na - pa}+{pb} = {c1} | "
                  f"ĐK2 (b<=t<a) = {nb - pb}+{pa} = {c2} | best={best}")
    for c in range(26):
        if fa[c] or fb[c]:
            c3 = na + nb - fa[c] - fb[c]
            best = min(best, c3)
            print(f"ĐK3 chữ {chr(97 + c)}: {na + nb} - {fa[c]} - {fb[c]} = {c3} | best={best}")
    return best

trace_prefix("aba", "caa")

fa (a..e): [2, 1, 0, 0, 0] | fb (a..e): [2, 0, 1, 0, 0]
t=a: a<=t: 2, b<=t: 2 | ĐK1 (a<=t<b) = 1+2 = 3 | ĐK2 (b<=t<a) = 1+2 = 3 | best=3
t=b: a<=t: 3, b<=t: 2 | ĐK1 (a<=t<b) = 0+2 = 2 | ĐK2 (b<=t<a) = 1+3 = 4 | best=2
t=c: a<=t: 3, b<=t: 3 | ĐK1 (a<=t<b) = 0+3 = 3 | ĐK2 (b<=t<a) = 0+3 = 3 | best=2
t=d: a<=t: 3, b<=t: 3 | ĐK1 (a<=t<b) = 0+3 = 3 | ĐK2 (b<=t<a) = 0+3 = 3 | best=2
t=e: a<=t: 3, b<=t: 3 | ĐK1 (a<=t<b) = 0+3 = 3 | ĐK2 (b<=t<a) = 0+3 = 3 | best=2
ĐK3 chữ a: 6 - 2 - 2 = 2 | best=2
ĐK3 chữ b: 6 - 1 - 0 = 5 | best=2
ĐK3 chữ c: 6 - 0 - 1 = 5 | best=2


2

In [13]:
trace_prefix("dabadd", "cda")

fa (a..e): [2, 1, 0, 3, 0] | fb (a..e): [1, 0, 1, 1, 0]
t=a: a<=t: 2, b<=t: 1 | ĐK1 (a<=t<b) = 4+1 = 5 | ĐK2 (b<=t<a) = 2+2 = 4 | best=4
t=b: a<=t: 3, b<=t: 1 | ĐK1 (a<=t<b) = 3+1 = 4 | ĐK2 (b<=t<a) = 2+3 = 5 | best=4
t=c: a<=t: 3, b<=t: 2 | ĐK1 (a<=t<b) = 3+2 = 5 | ĐK2 (b<=t<a) = 1+3 = 4 | best=4
t=d: a<=t: 6, b<=t: 3 | ĐK1 (a<=t<b) = 0+3 = 3 | ĐK2 (b<=t<a) = 0+6 = 6 | best=3
t=e: a<=t: 6, b<=t: 3 | ĐK1 (a<=t<b) = 0+3 = 3 | ĐK2 (b<=t<a) = 0+6 = 6 | best=3
ĐK3 chữ a: 9 - 2 - 1 = 6 | best=3
ĐK3 chữ b: 9 - 1 - 0 = 8 | best=3
ĐK3 chữ c: 9 - 0 - 1 = 8 | best=3
ĐK3 chữ d: 9 - 3 - 1 = 5 | best=3


3

Bảng tóm tắt ví dụ 1: ĐK1 tốt nhất ở `t = 'b'`: `a` không có chữ > `b`, `b` có 2 chữ `'a'` ≤ `'b'` ⇒ 2. ĐK3 chữ `'a'`: `6 − 2 − 2 = 2`. Đáp án `2`.

## 9. Code nộp LeetCode

In [14]:
class Solution:
    def minCharacters(self, a: str, b: str) -> int:
        fa = [0] * 26
        fb = [0] * 26
        for ch in a:
            fa[ord(ch) - 97] += 1
        for ch in b:
            fb[ord(ch) - 97] += 1
        na, nb = len(a), len(b)
        best = na + nb
        for c in range(26):                       # điều kiện 3
            best = min(best, na + nb - fa[c] - fb[c])
        pa = pb = 0
        for t in range(25):                       # ngưỡng 'a'..'y'
            pa += fa[t]
            pb += fb[t]
            best = min(best, (na - pa) + pb,      # điều kiện 1: a <= t < b
                             (nb - pb) + pa)      # điều kiện 2: b <= t < a
        return best

sol = Solution()
for (a, b), want in examples:
    got = sol.minCharacters(a, b)
    print(got, "OK" if got == want else "SAI")

2 OK
3 OK


## 10. Vì sao thuật toán đúng

**Điều kiện 1.** Giả sử sau khi đổi, `max(a') < min(b')`. Đặt `t = max(a')` ⇒ `t ≤ 'y'` (vì `min(b') > t`). Mọi vị trí `i` với `a[i] > t` đã bị đổi; mọi vị trí `j` với `b[j] ≤ t` đã bị đổi ⇒ chi phí `≥ cost1(t)`. Ngược lại, với mọi `t ∈ ['a', 'y']`, đổi đúng các ký tự vi phạm (ký tự của `a` thành `t`, ký tự của `b` thành `t+1`) cho ra trạng thái thỏa điều kiện 1 với chi phí `cost1(t)`. ⇒ Tối ưu điều kiện 1 = `min_t cost1(t)`.

**Điều kiện 2** đối xứng.

**Điều kiện 3.** Nếu cuối cùng mọi ký tự là `c`, mọi ký tự khác `c` phải đổi và chỉ chúng ⇒ `cost3(c)`; lấy min theo `c`.

Đáp án là min của ba điều kiện. Tổng tiền tố cho đúng `Σ_{≤t}` và `Σ_{>t} = len − Σ_{≤t}`.

**Kiểm chứng** ở mục 12: so với vét cạn **mọi chuỗi đích** trên bảng chữ cái thu nhỏ — vét cạn này không dùng lập luận ngưỡng.

## 11. Phân tích Time / Space

Gọi `N = |a| + |b| ≤ 2·10⁵`, `K = 26`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| Vét cạn chuỗi đích | `O(K^N · N)` | `O(N)` | chỉ kiểm chứng |
| 1. Thử ngưỡng, quét chuỗi | `O(K · N)` | `O(1)` | ~5·10⁶ phép Python |
| 2. Tần suất + cộng trực tiếp | `O(N + K²)` | `O(K)` | |
| 3. Tần suất + tiền tố | `O(N + K)` | `O(K)` | **nộp bài** |
| 4. `str.count` | `O(K · N)` (C) | `O(K)` | nhanh nhất thực tế |

In [15]:
for name, fn in [("scan", min_chars_scan), ("freq", min_chars_freq), ("prefix", min_chars_prefix),
                 ("count", min_chars_count), ("submit", Solution().minCharacters)]:
    t0 = time.perf_counter(); r = fn(A, B)
    print(f"{name:6s}: {r}  {time.perf_counter() - t0:.4f}s")

scan  : 99836  0.2700s
freq  : 99836  0.0074s
prefix: 99836  0.0079s
count : 99836  0.0031s
submit: 99836  0.0069s


## 12. Bộ test

1. Vét cạn mọi chuỗi đích với `K ∈ {2, 3, 4}`, độ dài ≤ 3 — so với các cách (cho tham số `K`).
2. Đề gốc (`K = 26`): đối chiếu chéo các cách trên chuỗi ngẫu nhiên.

In [16]:
fns_K = [min_chars_scan, min_chars_freq, min_chars_prefix]
fns_26 = fns_K + [min_chars_count, lambda a, b: Solution().minCharacters(a, b)]

for (a, b), want in examples:
    for fn in fns_26:
        assert fn(a, b) == want

random.seed(2024)
cnt = 0
for _ in range(600):
    K = random.choice([2, 3, 4])
    letters = "abcd"[:K]
    a = "".join(random.choice(letters) for _ in range(random.randint(1, 3)))
    b = "".join(random.choice(letters) for _ in range(random.randint(1, 3)))
    want = min_chars_exhaustive(a, b, K)
    for fn in fns_K:
        assert fn(a, b, K) == want, (fn, a, b, K)
    cnt += 1
print(f"{cnt} bộ: khớp vét cạn mọi chuỗi đích trên bảng chữ cái K ∈ {{2,3,4}}")

600 bộ: khớp vét cạn mọi chuỗi đích trên bảng chữ cái K ∈ {2,3,4}


In [17]:
edge = [(("a", "a"), 0),            # điều kiện 3 đã đúng
        (("a", "b"), 0),            # điều kiện 1 đã đúng
        (("z", "a"), 0),            # điều kiện 2 đã đúng
        (("z", "z"), 0),
        (("zz", "zz"), 0),
        (("az", "az"), 2),
        (("abc", "abc"), 3)]
for (a, b), want in edge:
    for fn in fns_26:
        assert fn(a, b) == want, (fn, a, b)

random.seed(7)
for _ in range(300):
    alpha = random.choice(["ab", "xyz", "az", "abcdefghijklmnopqrstuvwxyz"])
    a = "".join(random.choice(alpha) for _ in range(random.randint(1, 50)))
    b = "".join(random.choice(alpha) for _ in range(random.randint(1, 50)))
    res = {fn(a, b) for fn in fns_26}
    assert len(res) == 1, (a, b, res)
print("edge + 300 bộ K = 26: mọi cách khớp nhau")

edge + 300 bộ K = 26: mọi cách khớp nhau


Edge `("az", "az")`: ĐK1 với `t = 'a'`: `a` có `'z' > 'a'` (1), `b` có `'a' ≤ 'a'` (1) ⇒ 2; ĐK3: `4 − 2 = 2` ⇒ `2`. ✓

## 13. Lỗi người mới hay gặp

### Lỗi 1 — Cho ngưỡng chạy tới `'z'`

Với `t = 'z'`, điều kiện 1 đòi `b` có chữ `> 'z'` — không tồn tại; công thức `(|a| − PA) + PB` lại tính như thể `b` được "đổi" thành chữ không tồn tại.

In [18]:
def wrong_include_z(a, b):
    fa, fb = freq(a), freq(b); na, nb = len(a), len(b); best = na + nb; pa = pb = 0
    for t in range(26):                            # SAI: t = 'z' không hợp lệ
        pa += fa[t]; pb += fb[t]
        best = min(best, na - pa + pb, nb - pb + pa)
    for c in range(26):
        best = min(best, na + nb - fa[c] - fb[c])
    return best

a, b = "b", "zazb"
print("sai =", wrong_include_z(a, b), "| đúng =", Solution().minCharacters(a, b))

sai = 1 | đúng = 2


Với `t = 'z'`, công thức ĐK2 cho `(|b| − |b|) + |a| = |a|` — tức "đổi mọi ký tự của `a` thành chữ lớn hơn `'z'`", điều không thể. Ở ví dụ `a = "b"`, `b = "zazb"`, bản lỗi trả về `1` (đổi `'b'` của `a` thành chữ "> z") trong khi đáp án thật là `2`. **Sửa:** `t` chỉ chạy `0..24`.

### Lỗi 2 — Quên điều kiện 3

In [19]:
def wrong_no_cond3(a, b):
    fa, fb = freq(a), freq(b); na, nb = len(a), len(b); best = na + nb; pa = pb = 0
    for t in range(25):
        pa += fa[t]; pb += fb[t]
        best = min(best, na - pa + pb, nb - pb + pa)
    return best

for a, b in [("aaa", "aaa"), ("zzz", "zzz"), ("aaab", "bbba")]:
    print(f"{a},{b}: thiếu ĐK3 = {wrong_no_cond3(a, b)}, đúng = {Solution().minCharacters(a, b)}")

aaa,aaa: thiếu ĐK3 = 3, đúng = 0
zzz,zzz: thiếu ĐK3 = 3, đúng = 0
aaab,bbba: thiếu ĐK3 = 2, đúng = 2


`"zzz","zzz"` đã thỏa điều kiện 3 (đáp án 0) nhưng điều kiện 1/2 cần đổi 3 ký tự. **Sửa:** thêm `min_c (|a| + |b| − fa[c] − fb[c])`.

**Một quan sát thú vị (không phải lỗi):** phương án "đổi `a` thành toàn chữ phổ biến nhất của `a`, `b` thành toàn chữ phổ biến nhất của `b`" luôn **hợp lệ** — nếu hai chữ trùng nhau thì thỏa ĐK3, nếu khác nhau thì thỏa ĐK1 hoặc ĐK2. Cell dưới kiểm chứng rằng dùng phương án này thay cho ĐK3 vẫn cho đúng đáp án (vì ĐK3 tối ưu với chữ `c` không tốt hơn phương án này hoặc ĐK1/ĐK2).

In [20]:
def with_separate_majority(a, b):
    fa, fb = freq(a), freq(b)
    return min(wrong_no_cond3(a, b), (len(a) - max(fa)) + (len(b) - max(fb)))

random.seed(17)
for _ in range(3000):
    alpha = random.choice(["ab", "abc", "abcd", "xyz"])
    a = "".join(random.choice(alpha) for _ in range(random.randint(1, 8)))
    b = "".join(random.choice(alpha) for _ in range(random.randint(1, 8)))
    assert with_separate_majority(a, b) == Solution().minCharacters(a, b)
print("thay ĐK3 bằng 'mỗi chuỗi về chữ phổ biến nhất của nó' vẫn đúng trên 3000 bộ")

thay ĐK3 bằng 'mỗi chuỗi về chữ phổ biến nhất của nó' vẫn đúng trên 3000 bộ


### Lỗi 3 — Chỉ xét điều kiện 1 (quên đối xứng điều kiện 2)

In [21]:
def wrong_only_cond1(a, b):
    fa, fb = freq(a), freq(b); na, nb = len(a), len(b); best = na + nb; pa = pb = 0
    for t in range(25):
        pa += fa[t]; pb += fb[t]
        best = min(best, na - pa + pb)
    for c in range(26):
        best = min(best, na + nb - fa[c] - fb[c])
    return best

a, b = "xyz", "abc"
print("sai =", wrong_only_cond1(a, b), "| đúng =", Solution().minCharacters(a, b))

sai = 4 | đúng = 0


**Sửa:** xét cả `(|b| − PB) + PA`.

### Lỗi 4 — Dùng `<` thay vì `≤` khi tính "số ký tự của `b` phải đổi"

Với ngưỡng `t` (`a ≤ t < b`), ký tự `b[j] = t` **cũng** vi phạm (cần `> t`). Đếm `b[j] < t` thay vì `≤ t` ⇒ thiếu.

In [22]:
def wrong_strict(a, b):
    fa, fb = freq(a), freq(b); na, nb = len(a), len(b); best = na + nb
    for t in range(25):
        a_gt = sum(fa[t + 1:]); b_lt = sum(fb[:t])           # SAI: phải là fb[:t+1]
        best = min(best, a_gt + b_lt)
    for c in range(26):
        best = min(best, na + nb - fa[c] - fb[c])
    return best

a, b = "ab", "bc"
print("sai =", wrong_strict(a, b), "| đúng =", Solution().minCharacters(a, b))

sai = 0 | đúng = 1


Ở `("ab", "bc")`: bản lỗi với `t = 'b'` coi như `b = "bc"` không có ký tự vi phạm (chỉ đếm `< 'b'`) ⇒ trả về 0, nhưng `'b'` có mặt ở cả hai chuỗi nên điều kiện 1 chưa thỏa — đáp án thật là 1. **Sửa:** đếm `b[j] ≤ t`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Điều kiện "mọi phần tử nhóm A < mọi phần tử nhóm B" ⇒ **liệt kê ngưỡng** (chỉ có 25 ngưỡng với chữ cái).
- Chi phí chỉ phụ thuộc **tần suất** ⇒ nén chuỗi `10⁵` ký tự thành mảng 26 số.
- Đếm "≤ t" / "> t" nhiều lần ⇒ **tổng tiền tố**.
- Kiểm chứng bằng cách **thu nhỏ tham số** (bảng chữ cái `K = 3`) để brute force được.
- Bài cùng họ: các bài "đổi ít nhất để thỏa điều kiện sắp xếp" trên miền giá trị nhỏ.

## 15. Biến thể của bài

1. **Chỉ được sửa chuỗi `a`:** chi phí chỉ đếm phía `a`, nhưng `b` cố định ⇒ ngưỡng `t` phải `< min(b)` (ĐK1) và ĐK3 chỉ khả thi nếu `b` toàn một chữ.
2. **Chi phí đổi khác nhau theo chuỗi** (`ca` cho `a`, `cb` cho `b`): nhân trọng số trong công thức.
3. **Không nghiêm ngặt** (`≤` thay vì `<`): ngưỡng `t` cho phép cả `'z'`, và `a ≤ t ≤ b`.
4. **Ba chuỗi `a < b < c`:** hai ngưỡng `t1 < t2` ⇒ `O(K²)` cặp ngưỡng với tiền tố.

In [23]:
def min_chars_only_a(a, b):
    """Biến thể 1: chỉ sửa a."""
    fa = freq(a); na = len(a); INF = float("inf"); best = INF
    lo, hi = min(b), max(b)
    pa = 0
    for t in range(25):
        pa += fa[t]
        if chr(97 + t) < lo:                      # ĐK1: a <= t < min(b)
            best = min(best, na - pa)
        if chr(97 + t) >= hi:                     # ĐK2: max(b) <= t < a
            best = min(best, pa)
    if lo == hi:                                  # ĐK3: b toàn một chữ
        best = min(best, na - fa[ord(lo) - 97])
    return best if best != INF else -1

def three_strings(a, b, c):
    """Biến thể 4: a < b < c (mọi chữ), sửa cả ba."""
    fa, fb, fc = freq(a), freq(b), freq(c)
    PA = [0] + list(accumulate(fa)); PB = [0] + list(accumulate(fb)); PC = [0] + list(accumulate(fc))
    best = len(a) + len(b) + len(c)
    for t1 in range(25):
        for t2 in range(t1 + 1, 25):              # t2 <= 'y' để c còn chữ > t2
            # a <= t1, t1 < b <= t2, c > t2
            cost = (len(a) - PA[t1 + 1]) + (PB[t1 + 1] + (len(b) - PB[t2 + 1])) + PC[t2 + 1]
            best = min(best, cost)
    return best

print(min_chars_only_a("dabadd", "cda"), min_chars_only_a("zzz", "eee"), three_strings("ab", "ba", "ab"))

6 0 4


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| Vét cạn chuỗi đích | thử mọi `(a', b')` | mũ | `O(N)` | kiểm chứng với `K` nhỏ |
| 1. Quét theo ngưỡng | đếm trực tiếp trên chuỗi | `O(KN)` | `O(1)` | baseline |
| 2. Tần suất + cộng | `sum(f[..])` mỗi ngưỡng | `O(N + K²)` | `O(K)` | dễ viết |
| 3. Tần suất + tiền tố | cộng dồn | `O(N + K)` | `O(K)` | **nộp bài** |
| 4. `str.count` | đếm bằng C | `O(KN)` C | `O(K)` | nhanh thực tế |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `minCharacters("a", "z")`, `minCharacters("ace", "abe")`?

**Bài 2 (tính tay):** lập bảng `cost1(t)`, `cost2(t)` cho `t = 'a'..'d'` và `cost3` cho `a = "acd"`, `b = "bdd"`.

**Bài 3 (điền TODO):** hoàn thiện `best_condition(a, b)` trả về cặp `(chi phí, điều kiện số mấy, ngưỡng hoặc chữ)` của một phương án tối ưu.

**Bài 4 (tự viết lại):** viết lại Cách 3 với bảng chữ cái tham số `K` và kiểm chứng bằng `min_chars_exhaustive` cho `K = 3`.

**Bài 5 (mini challenge):** biến thể 4 (ba chuỗi `a < b < c`) — kiểm chứng `three_strings` bằng vét cạn chuỗi đích trên bảng chữ cái thu nhỏ `K = 4`, độ dài ≤ 2. (Biến thể tự đặt; cần tổng quát hóa `three_strings` theo `K`.)

In [24]:
def best_condition(a, b):
    # TODO: như Cách 3, nhưng ghi lại (cost, condition, param) tốt nhất
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) `"a" < "z"` đã thỏa ĐK1 ⇒ 0. (2) `"ace"` vs `"abe"`: ĐK3 cần đổi 4; ĐK1 với `t='c'` cần 3. (3) ĐK2 với `t='b'`: `b` phải `≤ 'b'` (đổi `'e'`), `a` phải `> 'b'` (đổi `'a'`) ⇒ **2** — đừng quên điều kiện đối xứng!

**Bài 2:** (1) `fa: a1 c1 d1`, `fb: b1 d2`. (2) `t='c'`: ĐK1 = `1 (d) + 1 (b) = 2`; ĐK2 = `2 (d,d) + 2 (a,c) = 4`. (3) ĐK3 chữ `d`: `6 − 1 − 2 = 3`. Đáp án `2`.

**Bài 3:** (1) Giữ `best = (inf, None, None)`. (2) So sánh tuple theo chi phí. (3) Với ngưỡng trả về `chr(97 + t)`.

**Bài 4:** (1) Đổi `range(25)` thành `range(K − 1)`, `range(26)` thành `range(K)`. (2) `freq(s, K)`. (3) Vòng kiểm thử như mục 12.

**Bài 5:** (1) Thêm tham số `K` cho `three_strings`. (2) Vét cạn: `product(letters, repeat=...)` cho ba chuỗi, kiểm tra `max(a') < min(b')` và `max(b') < min(c')`. (3) So sánh trên 200 bộ.

## 19. Đáp án tham khảo

In [25]:
# Bài 1
assert Solution().minCharacters("a", "z") == 0 and Solution().minCharacters("ace", "abe") == 2

# Bài 2
assert Solution().minCharacters("acd", "bdd") == 2

# Bài 3
def best_condition_ans(a, b):
    fa, fb = freq(a), freq(b); na, nb = len(a), len(b)
    best = (na + nb + 1, None, None)
    for c in range(26):
        best = min(best, (na + nb - fa[c] - fb[c], 3, chr(97 + c)))
    pa = pb = 0
    for t in range(25):
        pa += fa[t]; pb += fb[t]
        best = min(best, (na - pa + pb, 1, chr(97 + t)), (nb - pb + pa, 2, chr(97 + t)))
    return best

for (a, b), want in examples:
    cost, cond, param = best_condition_ans(a, b)
    assert cost == want
    print(f"{a},{b}: chi phí {cost}, điều kiện {cond}, tham số {param!r}")

# Bài 4 — đã có min_chars_prefix(a, b, K); kiểm chứng K = 3
random.seed(41)
for _ in range(200):
    a = "".join(random.choice("abc") for _ in range(random.randint(1, 3)))
    b = "".join(random.choice("abc") for _ in range(random.randint(1, 3)))
    assert min_chars_prefix(a, b, 3) == min_chars_exhaustive(a, b, 3)
print("Bài 1-4: OK")

aba,caa: chi phí 2, điều kiện 1, tham số 'b'
dabadd,cda: chi phí 3, điều kiện 1, tham số 'd'
Bài 1-4: OK


In [26]:
# Bài 5 — ba chuỗi a < b < c, tổng quát hóa theo K và kiểm chứng vét cạn
def three_strings_K(a, b, c, K):
    fa, fb, fc = freq(a, K), freq(b, K), freq(c, K)
    PA = [0] + list(accumulate(fa)); PB = [0] + list(accumulate(fb)); PC = [0] + list(accumulate(fc))
    best = None
    for t1 in range(K):
        for t2 in range(t1 + 1, K - 1):
            cost = (len(a) - PA[t1 + 1]) + PB[t1 + 1] + (len(b) - PB[t2 + 1]) + PC[t2 + 1]
            best = cost if best is None else min(best, cost)
    return best

def three_exhaustive(a, b, c, K):
    L = [chr(97 + i) for i in range(K)]; best = None
    for a2 in product(L, repeat=len(a)):
        for b2 in product(L, repeat=len(b)):
            if max(a2) >= min(b2): continue
            for c2 in product(L, repeat=len(c)):
                if max(b2) < min(c2):
                    cost = sum(x != y for x, y in zip(a + b + c, a2 + b2 + c2))
                    best = cost if best is None else min(best, cost)
    return best

random.seed(42)
for _ in range(200):
    K = 4
    a, b, c = ("".join(random.choice("abcd") for _ in range(random.randint(1, 2))) for _ in range(3))
    assert three_strings_K(a, b, c, K) == three_exhaustive(a, b, c, K), (a, b, c)
assert three_strings_K("ab", "ba", "ab", 26) == three_strings("ab", "ba", "ab")
print("Bài 5: ba chuỗi khớp vét cạn với K = 4; ví dụ ('ab','ba','ab') ->", three_strings_K("ab", "ba", "ab", 26))

Bài 5: ba chuỗi khớp vét cạn với K = 4; ví dụ ('ab','ba','ab') -> 4


Lưu ý Bài 5: ngưỡng `t2` chỉ chạy tới `K − 2` để `c` còn chữ `> t2`, và `t1 < t2` để `b` có chữ trong `(t1, t2]` — cùng tinh thần với Lỗi 1 (ngưỡng không được là chữ cuối bảng).

## 20. Template nhớ nhanh + Checklist

```python
fa, fb = freq(a), freq(b); na, nb = len(a), len(b)
best = min(na + nb - fa[c] - fb[c] for c in range(26))     # ĐK3: cùng một chữ
pa = pb = 0
for t in range(25):                                         # ngưỡng 'a'..'y'
    pa += fa[t]; pb += fb[t]                                # số ký tự <= t
    best = min(best, (na - pa) + pb,                        # ĐK1: a <= t < b
                     (nb - pb) + pa)                        # ĐK2: b <= t < a
return best
```

**Checklist trước khi submit:**
- [ ] Ngưỡng chỉ `'a'..'y'` (25 giá trị).
- [ ] Đủ **ba** điều kiện, ĐK2 là đối xứng của ĐK1.
- [ ] ĐK3 dùng **cùng** một chữ cho cả hai chuỗi.
- [ ] Ký tự bằng ngưỡng: thuộc phía `≤ t`.
- [ ] Thử các trường hợp đã thỏa sẵn (đáp án 0).